# TP2 — Binôme A (extraction de features)


## Setup


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("donnees")


## Extraction des features


In [ ]:
def load_pdw(pdw):
    # Charge un .npz et trie par date
    date = np.array(pdw["date"], dtype=float)
    order = np.argsort(date)
    return {
        "date": date[order],
        "largeur": np.array(pdw["largeur"], dtype=float)[order],
        "frequence": np.array(pdw["frequence"], dtype=float)[order],
        "puissance": np.array(pdw["puissance"], dtype=float)[order],
        "theta": np.array(pdw["theta"], dtype=float)[order],
        "phi": np.array(pdw["phi"], dtype=float)[order],
    }


def clean_features(feats):
    # Remplace NaN / Inf par 0
    clean = {}
    for k, v in feats.items():
        v = float(v)
        if np.isnan(v) or np.isinf(v):
            clean[k] = 0.0
        else:
            clean[k] = v
    return clean


def extract_features_avancees(pdw):
    # Features Binôme A : PRI, ranges d'angles, spans, contraste de puissance
    d = load_pdw(pdw)
    date = d["date"]
    largeur = d["largeur"]
    frequence = d["frequence"]
    puissance = d["puissance"]
    theta = d["theta"]
    phi = d["phi"]

    pri = np.diff(date)
    pri = pri[pri > 0]

    feats = {
        "pri_median": float(np.median(pri)) if len(pri) else 0.0,
        "pri_min": float(np.min(pri)) if len(pri) else 0.0,
        "pri_max": float(np.max(pri)) if len(pri) else 0.0,
        "theta_range": float(np.max(theta) - np.min(theta)),
        "phi_range": float(np.max(phi) - np.min(phi)),
        "frequence_span": float(np.max(frequence) - np.min(frequence)),
        "largeur_span": float(np.max(largeur) - np.min(largeur)),
        "puissance_peak_to_mean": float(np.max(puissance) - np.mean(puissance)),
    }
    return clean_features(feats)


# test rapide
demo = np.load(DATA_DIR / "train" / "pdw-0.npz")
print(extract_features_avancees(demo))


## Chargement du dataset


In [ ]:
def build_dataset(data_dir, labels_path):
    # Construit X (features) et y (labels) pour un split
    with open(labels_path, encoding="utf-8") as f:
        labels = json.load(f)

    rows = []
    index = []
    y_list = []

    for name, label in labels.items():
        pdw = np.load(Path(data_dir) / f"{name}.npz")
        rows.append(extract_features_avancees(pdw))
        index.append(name)
        y_list.append(1 if label == "menace" else 0)

    X = pd.DataFrame(rows, index=index)
    y = pd.Series(y_list, index=index, name="y")
    return X, y


X_train, y_train = build_dataset(DATA_DIR / "train", DATA_DIR / "train_labels.json")
print("X_train :", X_train.shape)
print(y_train.value_counts())
X_train.head()


## Représentation des features


In [ ]:
# Répartition des classes
print(y_train.value_counts())

plt.figure(figsize=(5, 3))
plt.bar(["nonmenace", "menace"], [
    (y_train == 0).sum(),
    (y_train == 1).sum(),
])
plt.title("Classes (train)")
plt.ylabel("nombre de signaux")
plt.show()


In [ ]:
# Histogrammes menace vs nonmenace
features = list(X_train.columns)

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.ravel()

for ax, feat in zip(axes, features):
    ax.hist(X_train.loc[y_train == 1, feat], bins=30, alpha=0.5, label="menace")
    ax.hist(X_train.loc[y_train == 0, feat], bins=30, alpha=0.5, label="nonmenace")
    ax.set_title(feat)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()


In [ ]:
# Boxplots par feature
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.ravel()

for ax, feat in zip(axes, features):
    data = [
        X_train.loc[y_train == 1, feat],
        X_train.loc[y_train == 0, feat],
    ]
    ax.boxplot(data, labels=["menace", "nonmenace"])
    ax.set_title(feat)

plt.tight_layout()
plt.show()
